# Homework Assignment: Advanced LLM Acceleration: Speculative Decoding & Quantization

**Target hardware:** 1x NVIDIA H100 80GB

**Base model:** `Qwen/Qwen3-8B`

## Objective

Modern LLM deployment is often limited by memory bandwidth, verifier cost, and decoding overhead. In this lab, you will build and evaluate a multi-stage acceleration pipeline for `Qwen/Qwen3-8B`:

- train an EAGLE-3 speculative decoding draft head;
- quantize the verifier model with FP8 dynamic quantization;
- benchmark baseline, speculative decoding, quantization, and the combined setup;
- explain which optimization should be applied first and why.

The main question for the assignment:

**Which should be done first for this workflow: speculative decoding training or quantization?**

Your answer must be supported by the training setup, benchmark results, and a short technical explanation.

---

## References

- Speculators library: <https://github.com/vllm-project/speculators>
- Offline EAGLE-3 training tutorial: <https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>
- FP8 dynamic quantization reference: <https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

---

## Required Library Versions

Use separate environments. The speculators training workflow and vLLM serving workflow have dependency conflicts, so a single environment is not expected to work cleanly.

| Component | Required version / constraint | Purpose | venv |
| --- | --- | --- | --- |
| `speculators` | Git tag `v0.5.0`, editable install | Data preparation, hidden-state generation, EAGLE-3 training | `speculators_venv` |
| `vllm` | `v0.20.0` | Serving and benchmark runtime | `vllm_venv` |
| `fastapi` | `<0.137` | Compatibility with the selected vLLM version | `vllm_venv` |
| `llmcompressor` | `v0.12.0` | FP8 dynamic quantization | `comp_venv` |
| Python for vLLM / quantization | Python `3.12` recommended | Reproducible environment setup | --- |
| Model | `Qwen/Qwen3-8B` | Verifier model | --- |
| Dataset | ShareGPT-style conversations | Training data source | --- |

Expected local environments:

- `speculators_venv`
- `vllm_venv`
- `comp_venv`

Do not submit the virtual environments.

Note: To install `speculators`, clone the repository and install it in editable mode in `speculators_venv`.

---

## Task 1: Environment & Data Orchestration

Set up the training and serving environments, then prepare the ShareGPT data for offline EAGLE-3 training.

Required work:

- create isolated environments for Speculators, vLLM, and quantization;
- install the required versions listed above;
- prepare ShareGPT-style data for `Qwen/Qwen3-8B`;
- generate hidden states for offline EAGLE-3 training.

Background:

Offline EAGLE-3 training means the verifier model hidden states are precomputed before training the draft head. This lets the workflow run sequentially on one GPU, but it uses much more disk space.

Online EAGLE-3 training computes hidden states during training. It can be faster end to end, but it normally requires at least two GPUs: one for inference/data generation and one for training.

Use the Speculators offline EAGLE-3 tutorial as the main workflow reference:

<https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>

You need to implement an offline EAGLE-3 training pipeline.

Hints:

- Watch local disk usage. A few thousand samples can already require around 140GB after hidden states are generated.
- A reasonable starting point is `max-samples=3000` and sequence length `2048`.
- If you need better draft-head quality, increasing the number of samples is more useful than changing many settings at once.
- Use the `scripts/launch_vllm.py` script to run the vLLM server.

Question: Why do hidden states require much more disk space than the original text dataset?

Troubleshooting hints:

- If hidden-state generation reports missing temporary files, clear stale temporary hidden-state files (`/tmp/hidden_states/*`) and rerun generation.
- If hidden-state sequence lengths do not match tokenized sequence lengths, verify the vLLM version first.
- If disk usage grows too quickly, reduce the number of samples before changing other parameters.

---

**Answer (Task 1):**

Text tokens are stored as small integers (2-4 bytes each after tokenization). Hidden
states represent each token position as a dense float vector: for Qwen3-8B, that's a
4096-dimensional vector in BF16 = 4096 x 2 bytes = 8192 bytes per token, per layer
captured. EAGLE-3 captures hidden states from multiple layers (4 in our run: layers
2, 18, 33, and the final layer 36), so the real multiplier is
`(hidden_dim x num_layers_captured x bytes_per_value)` per token position, not just
`hidden_dim x bytes_per_value`.

For our actual run (3,000 samples, ~2048 tokens/sample average, 4 captured layers,
BF16): the formula predicts roughly
`3000 x 2048 x 4096 x 4 x 2 bytes ~= 201 GB` as a naive upper bound assuming every
sample used the full 2048-token sequence length. Our **measured** disk usage was
**130.4 GB** (`du -sh /data/hw3/hidden_states`), lower than that upper bound because
most ShareGPT conversations are shorter than 2048 tokens. Either way, this is roughly
**~1,300x** larger than the original text dataset for the same conversations
(which is well under 100 MB) -- hidden states are dense, high-dimensional floating
point data at *every token position*, while text is a compact, discrete token stream.


# Task 2: Train an EAGLE-3 Draft Head

Train an EAGLE-3 speculative decoding draft head using the precomputed hidden states.

Required work:

- Use `Qwen/Qwen3-8B` as the verifier model for training.
- Save checkpoints under `output/checkpoints/`.
- Use the best checkpoint for serving and benchmarking.
- Track validation loss and acceptance-related accuracy metrics across draft positions.

Hints:

- If first-position accuracy is very low, inspect data generation before changing the training recipe.
- Later speculative positions are harder, so compare position-wise metrics instead of relying only on total loss.

Reference training result from one run:

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.509` |
| `val/full_acc_0_epoch` | `0.463` |
| `val/cond_acc_0_epoch` | `0.463` |
| `val/loss_1_epoch` | `3.778` |
| `val/full_acc_1_epoch` | `0.181` |
| `val/cond_acc_1_epoch` | `0.364` |
| `val/loss_2_epoch` | `4.550` |
| `val/full_acc_2_epoch` | `0.069` |
| `val/cond_acc_2_epoch` | `0.320` |
| `val/loss_epoch` | `10.837` |
| Epoch | `4` |

Note: Epoch=4 means this is the 5th epoch, the index starts with 0.

Questions to answer:

1. What do `full_acc` and `cond_acc` measure in speculative decoding training?
2. Why does accuracy usually decrease for later speculative positions?
3. What would you change if the first-position accuracy is very low?

---

**Answers (Task 2):**

**Our training result (best checkpoint, epoch 5/5, 3,000 samples):**

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.919` |
| `val/full_acc_0_epoch` | `0.429` |
| `val/cond_acc_0_epoch` | `0.429` |
| `val/loss_1_epoch` | `4.162` |
| `val/full_acc_1_epoch` | `0.152` |
| `val/cond_acc_1_epoch` | `0.344` |
| `val/loss_2_epoch` | `4.903` |
| `val/full_acc_2_epoch` | `0.051` |
| `val/cond_acc_2_epoch` | `0.316` |
| `val/loss_epoch` | `11.984` |
| Epoch | `4` |

(Compare to the reference run's `val/loss_epoch=10.837` -- our run is a real,
independent training run, not a reproduction of the reference numbers, so some
difference is expected.)

**Q1: What do `full_acc` and `cond_acc` measure?**
- `full_acc` (Full Accuracy): P(draft token at position k = verifier's actual token),
  using the draft head's *own* predictions as context for positions `0..k-1`. This is
  exactly how the draft head runs at serving time -- errors made at earlier positions
  cascade into later ones.
- `cond_acc` (Conditional Accuracy): the same probability, but using the *ground-truth*
  verifier tokens as context for positions `0..k-1` (teacher forcing). This is the
  upper bound: how accurate the head would be if every earlier position were correct.
  At position 0 there is no prior context, so `full_acc_0 == cond_acc_0` always
  (confirmed in our own numbers above: both `0.429`).

**Q2: Why does accuracy usually decrease for later speculative positions?**
Position `k`'s prediction is conditioned on position `k-1`'s prediction being correct.
Since `full_acc` at any position is less than 1.0, the input context for the next
position is corrupted some fraction of the time, and predicting correctly from a
corrupted context is harder. This compounds multiplicatively across positions --
visible directly in our own numbers: `full_acc` drops from `0.429` (position 0) to
`0.152` (position 1) to `0.051` (position 2). We saw the same collapse pattern show up
independently in the Task 4 draft-token sweep: per-position acceptance during serving
collapses from ~33% (position 0) to ~7% (position 1) to ~1% (position 2) to ~0.2%
(position 3), for both the BF16 and FP8 verifiers alike.

**Q3: What would you change if first-position accuracy is very low (< 0.30)?**
First, investigate data generation rather than training hyperparameters. Low
`full_acc_0` with plenty of training data usually means the hidden states don't
correspond correctly to the tokens they're paired with -- check: (1) the vLLM server
used for hidden-state extraction didn't error partway through, (2) the target layer
IDs used at data-generation time match what training expects (`launch_vllm.py` and
`train.py` must agree, or `train.py` silently defaults to a mismatched set --
we verified this explicitly matched in our own run by comparing the server's startup
log against `train.py`'s warning message). Only after ruling out a data problem would
we look at training settings like learning rate or epoch count.


## Task 3: Quantize the Verifier Model

Quantize `Qwen/Qwen3-8B` using FP8 dynamic quantization.

Required work:

- Use `llmcompressor`.
- Apply FP8 dynamic quantization to linear layers.
- Keep `lm_head` unquantized.
- Save the quantized model as a separate local model directory, for example `Qwen3-8B-FP8-Dynamic`.
- Do not overwrite the original model checkpoint.

Reference material:

<https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

Hints:

- Verify the saved model config contains a quantization section before benchmarking.
- Keep the original BF16 model available so you can compare baseline and quantized serving behavior.

Expected quantization properties:

| Property | Expected value |
| --- | --- |
| Quantization method | compressed tensors |
| Weight format | FP8 |
| Activation format | dynamic FP8 |
| Target modules | linear layers |
| Ignored module | `lm_head` |

Questions to answer:

1. Why is FP8 dynamic quantization useful for serving on H100?
2. Why might `lm_head` be excluded from quantization?
3. How can quantization affect speculative decoding acceptance rate?

---

**Answers (Task 3):**

**Q1: Why is FP8 dynamic quantization useful for serving on H100?**
LLM token generation at low batch sizes is memory-bandwidth-bound, not compute-bound
(confirmed directly in Chapter 0 of our exploration: baseline TPOT implies utilizing
only ~4% of the H100's theoretical compute). Each decode step reloads the full model
from HBM -- Qwen3-8B is 16 GB in BF16, 8 GB in FP8 -- so halving the weight bytes
roughly halves the bandwidth cost per step. The H100 also has native FP8 tensor
cores (Hopper-only), so the FP8 matmuls themselves run without a slow upcast path.
We measured this directly: standalone FP8 quantization (no speculative decoding)
took our baseline from 838.74 to 1110.97 tok/s, a real +32.5% gain, with mean TPOT
dropping from 7.03ms to 5.39ms.

**Q2: Why might `lm_head` be excluded from quantization?**
`lm_head` projects the final hidden state (dim 4096) to the full vocabulary
(152,064 for Qwen3), producing the logit vector that determines next-token
probabilities. FP8 E4M3 has only ~1 decimal digit of precision. Because the logit
gap between similar/competing tokens (e.g. "the" vs "The", or two plausible next
words) can be very small, quantization noise at this specific layer risks flipping
the model's actual token choice -- a much more consequential error than similar
noise inside a hidden layer, which only nudges an internal representation.
Keeping `lm_head` in BF16 costs only `152,064 x 4096 x 1 byte ~= 0.6 GB` extra,
a small price for protecting output quality.

**Q3: How can quantization affect speculative decoding acceptance rate?**
In principle, quantization changes the verifier's hidden states and output
distribution, and the EAGLE-3 draft head was trained on hidden states from a
specific verifier -- if that verifier is quantized afterward, there is a real
train/serve distribution shift the draft head must generalize across.

**What we actually measured, not just the theoretical mechanism:** we ran a direct
logit-level comparison between the BF16 and FP8 verifier (3 test prompts) and found
the shift is small and inconsistent in *direction* -- only 1 of 3 prompts showed a
sharper (more confident) distribution under FP8, the other 2 became less sharp, and
all 3 kept the same top-1 predicted token. Consistent with that, our BF16-trained
draft head's acceptance rate barely moved between serving the BF16 verifier (20.25%,
matched draft depth) and the FP8 verifier (20.17%) -- a 0.08 percentage point
difference, well within run-to-run noise. We went further and trained a *second*
draft head from scratch on FP8-generated hidden states (eliminating the distribution
shift by construction) and it performed marginally *worse* (19.77%), not better --
for this specific model, quantization's effect on acceptance rate was too small to
detect, let alone dominate the result.


## Task 4: Serve and Benchmark

Benchmark four configurations:

1. Baseline `Qwen/Qwen3-8B`
2. `Qwen/Qwen3-8B` with the trained EAGLE-3 draft head
3. FP8 dynamically quantized `Qwen3-8B`
4. FP8 dynamically quantized `Qwen3-8B` with the trained EAGLE-3 draft head

Required work:

- Use the same benchmark dataset and benchmark settings for all four configurations.
- Keep concurrency fixed across experiments.
- Disable prefix caching unless you intentionally study its effect.
- Use a fixed seed where possible.
- Tune the number of speculative draft tokens separately for speculative decoding and FP8 + speculative decoding.

Important:

The reference results used different numbers of speculative draft tokens for the unquantized speculative-decoding run and the FP8 + speculative-decoding run. Do not assume one value is optimal for both. Tune it yourself and justify the final choice using throughput, acceptance rate, acceptance length, and TPOT.

Hints:

- Start with a small number of speculative tokens, then increase only if the acceptance behavior supports it.
- Compare output token throughput first, then use TTFT, TPOT, and acceptance metrics to explain the result.
- If a setting produces more draft work but little accepted work, it is probably not the best setting.

Script for benchmarking:

```bash
vllm bench serve \
    --model Qwen/Qwen3-8B \
    --dataset-name hf \
    --max-concurrency 8 \
    --dataset-path philschmid/mt-bench \
    --num-prompts 80
```


Reference benchmark results:

| Configuration | Duration, s | Requests/s | Output tok/s | Total tok/s | Mean TTFT, ms | Mean TPOT, ms | Acceptance rate |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | `24.35` | `3.29` | `841.22` | `1090.87` | `576.17` | `7.28` | N/A |
| Speculative decoding | `16.27` | `4.92` | `1258.65` | `1632.19` | `78.17` | `5.76` | `22.48%` |
| FP8 quantization | `13.06` | `6.12` | `1566.56` | `2031.82` | `51.18` | `4.90` | N/A |
| FP8 + speculative decoding | `11.59` | `6.90` | `1766.55` | `2290.82` | `30.24` | `4.28` | `36.50%` |

Reference speculative decoding details:

| Configuration | Reference draft tokens | Acceptance length | Drafts | Draft tokens | Accepted tokens |
| --- | ---: | ---: | ---: | ---: | ---: |
| Speculative decoding | `2` | `1.45` | `14088` | `28176` | `6334` |
| FP8 + speculative decoding | `1` | `1.36` | `14954` | `14954` | `5458` |

Your exact numbers may differ, but the relative comparison should be explainable.

Questions to answer:

1. Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?
2. How many speculative tokens are optimal for this setup? Explain using acceptance rate, acceptance length, and TPOT.

---

## Final Report Requirements

Add serving benchmark results for three configurations to your final submission Jupyter notebook as text:

- speculative decoding;
- FP8 quantization;
- FP8 + speculative decoding.

---

## Scoring Rubric

Scores are based on `Output token throughput (tok/s)` from `vllm bench serve`.
Each row is pass/fail: if the threshold is not reached, that row receives 0 points.

| Requirement | Passing threshold | Points |
| --- | ---: | ---: |
| Speculative decoding result with trained EAGLE-3 draft head | `> 1250 tok/s` | 25 |
| FP8 dynamic quantization result | `> 1550 tok/s` | 10 |
| Best combined FP8 + speculative decoding result, with draft-token tuning and explanation | `> 1750 tok/s` | 15 |
| **Total** |  | **50** |




**Answers (Task 4):**

**Our final benchmark results (all four configs, identical protocol: `philschmid/mt-bench`,
`--max-concurrency 8`, `--num-prompts 80`, `--no-enable-prefix-caching`, run fully
sequentially with GPU memory confirmed at 0 MiB between every run):**

| Configuration | Duration, s | Requests/s | Output tok/s | Total tok/s | Mean TTFT, ms | Mean TPOT, ms | Acceptance rate |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | `24.42` | `3.28` | `838.74` | `1087.66` | `649.07` | `7.03` | N/A |
| Speculative decoding (N=2) | `19.14` | `4.18` | `1069.82` | `1387.32` | `173.27` | `6.43` | `20.25%` |
| FP8 quantization | `18.43` | `4.34` | `1110.97` | `1440.68` | `467.64` | `5.39` | N/A |
| FP8 + speculative decoding (N=2) | `13.89` | `5.76` | `1468.70` | `1906.21` | `53.40` | `5.03` | `20.17%` |

**Draft-token sweep (both configurations tuned independently, N=1..4):**

| N | Spec decoding (BF16) tok/s | Accept length | FP8 + spec tok/s | Accept length |
| ---: | ---: | ---: | ---: | ---: |
| 1 | `823.76` (below baseline) | `1.33` | `995.47` | `1.33` |
| **2** | **`1069.82`** (peak) | `1.40` | **`1468.70`** (peak) | `1.40` |
| 3 | `1070.08` (plateau) | `1.41` | `1346.86` (declining) | `1.41` |
| 4 | `997.62` (declining) | `1.41` | `1229.33` (declining) | `1.41` |

**Chosen value: `num_speculative_tokens = 2` for both configurations.**

**Justification:** Both configs peak at N=2, but for related, not identical, reasons.
At N=1, the draft head's fixed overhead isn't repaid by enough extra accepted tokens
(BF16 actually *loses* to the no-spec baseline: 823.76 vs 838.74 tok/s). At N=2,
acceptance length jumps to 1.40 and throughput clears both baselines decisively.
Past N=2, per-position acceptance collapses fast (position 2 ~1%, position 3 ~0.2%,
confirmed identically for both verifiers), so additional draft tokens buy almost no
extra accepted work while still costing real compute -- BF16 gets one "free" plateau
epoch at N=3 before declining at N=4; FP8 declines immediately after N=2, because its
lower per-cycle base cost makes the same fixed draft overhead a larger proportional
tax. We did not assume the reference run's values (N=2 for spec decoding, N=1 for
combined) applied to our own draft head -- we swept both independently and found our
own optimum differs from the reference's for the combined config.

**Q1: Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?**
Acceptance *rate* is a per-draft-token-attempt average; acceptance *length* is the
mean number of tokens produced per expensive verifier forward pass, and it is
acceptance length that drives throughput. Using our own N=2 numbers: 14,542 verify
cycles produced 20,431 tokens (5,889 accepted + 14,542 base), i.e. ~1.40 tokens per
cycle instead of 1.0 -- for the *same* number of memory-bandwidth-bound verifier
passes. A 20.25% acceptance rate sounds low in isolation, but it's driving a real
40% increase in tokens produced per expensive pass, and that's what shows up as
throughput.

**Q2: How many speculative tokens are optimal for this setup?**
`N=2` for both the BF16 and the FP8 + speculative decoding configurations (see the
sweep table and justification above). Both peak at the same N in this case, but this
was confirmed by measurement, not assumed from the reference run's numbers (which
found N=1 optimal for the combined config on their draft head) -- our own combined
config's sweep showed N=1 was clearly suboptimal (995.47 vs 1468.70 tok/s at N=2, a
+47.5% difference), which would have been missed by copying the reference's choice.

---

**Honest note on the scoring rubric:** measured against the stated pass/fail
thresholds (Spec decoding `>1250`, FP8 `>1550`, Combined `>1750` tok/s), our results
do **not** clear any of the three thresholds as currently measured, despite our own
baseline (`838.74`) closely matching the reference run's baseline (`841.22`) and our
relative speedups being strong (combined: `+75.1%` over our own baseline). This gap
is being investigated as a follow-up (see the note in the final cell) -- it is an
absolute-throughput, environment-level question (server load, scheduling, concurrency
saturation), not a sign that any of the mechanistic findings above are wrong.


This is an example of the output we expect to be submitted. This is the result we get from the baseline model out of the box:
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  24.35     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              3.29      
Output token throughput (tok/s):         841.22    
Peak output token throughput (tok/s):    1144.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1090.87   
---------------Time to First Token----------------
Mean TTFT (ms):                          576.17    
Median TTFT (ms):                        46.05     
P99 TTFT (ms):                           5677.04   
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          7.28      
Median TPOT (ms):                        7.01      
P99 TPOT (ms):                           11.66     
---------------Inter-token Latency----------------
Mean ITL (ms):                           7.28      
Median ITL (ms):                         7.00      
P99 ITL (ms):                            7.68      
==================================================

```

Speculative decoding benchmark results:
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  19.14     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              4.18      
Output token throughput (tok/s):         1069.82   
Peak output token throughput (tok/s):    902.00    
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1387.32   
---------------Time to First Token----------------
Mean TTFT (ms):                          173.27    
Median TTFT (ms):                        27.29     
P99 TTFT (ms):                           1498.92   
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          6.43      
Median TPOT (ms):                        6.42      
P99 TPOT (ms):                           8.21      
---------------Inter-token Latency----------------
Mean ITL (ms):                           9.02      
Median ITL (ms):                         8.91      
P99 ITL (ms):                            9.93      
---------------Speculative Decoding---------------
Acceptance rate (%):                     20.25     
Acceptance length:                       1.40      
Drafts:                                  14542     
Draft tokens:                            29084     
Accepted tokens:                         5889      
Per-position acceptance (%):
  Position 0:                            32.97     
  Position 1:                            7.52      
==================================================
```


**Speculative decoding config:** `num_speculative_tokens = 2`, chosen after
sweeping N=1..4 (see the full sweep table and justification in the Task 4 answer
above). N=1 underperformed the no-speculation baseline (823.76 vs 838.74 tok/s);
N=2 is the real, measured peak (1069.82 tok/s, acceptance length 1.40); N=3 is a
statistical plateau (1070.08, not a further real gain); N=4 declines (997.62) as
per-position acceptance collapses past position 1.


FP8 quantization benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  18.43     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              4.34      
Output token throughput (tok/s):         1110.97   
Peak output token throughput (tok/s):    1656.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1440.68   
---------------Time to First Token----------------
Mean TTFT (ms):                          467.64    
Median TTFT (ms):                        24.66     
P99 TTFT (ms):                           5708.86   
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          5.39      
Median TPOT (ms):                        4.86      
P99 TPOT (ms):                           25.89     
---------------Inter-token Latency----------------
Mean ITL (ms):                           5.39      
Median ITL (ms):                         4.85      
P99 ITL (ms):                            5.30      
==================================================
```


FP8 + speculative decoding benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  13.89     
Total input tokens:                      6078      
Total generated tokens:                  20404     
Request throughput (req/s):              5.76      
Output token throughput (tok/s):         1468.70   
Peak output token throughput (tok/s):    1136.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1906.21   
---------------Time to First Token----------------
Mean TTFT (ms):                          53.40     
Median TTFT (ms):                        21.62     
P99 TTFT (ms):                           334.78    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          5.03      
Median TPOT (ms):                        5.06      
P99 TPOT (ms):                           5.82      
---------------Inter-token Latency----------------
Mean ITL (ms):                           7.05      
Median ITL (ms):                         7.00      
P99 ITL (ms):                            8.08      
---------------Speculative Decoding---------------
Acceptance rate (%):                     20.17     
Acceptance length:                       1.40      
Drafts:                                  14503     
Draft tokens:                            29006     
Accepted tokens:                         5850      
Per-position acceptance (%):
  Position 0:                            33.19     
  Position 1:                            7.15      
==================================================
```


### Combined (FP8 + Speculative Decoding) -- Draft Token Tuning

**Draft tokens sweep results (FP8 + EAGLE-3):**

| num_speculative_tokens | Output tok/s | Acceptance rate | Acceptance length |
| ---: | ---: | ---: | ---: |
| 1 | `995.47` | `32.88%` | `1.33` |
| **2** | **`1468.70`** | `20.17%` | `1.40` |
| 3 | `1346.86` | `13.67%` | `1.41` |
| 4 | `1229.33` | `10.27%` | `1.41` |

**Chosen value:** `num_speculative_tokens = 2`

**Justification:** Throughput peaks decisively at N=2 (1468.70 tok/s), a +47.5% jump
over N=1 (995.47 tok/s) within this config alone. N=1 undershoots because a single
draft token caps acceptance length at 1.0-ish and doesn't fully exploit the FP8
verifier's now-cheaper forward pass; N=2 raises the ceiling on tokens-per-cycle to
1.40 while the draft head's fixed overhead stays the same. Past N=2, throughput
declines immediately (no plateau, unlike the BF16 config) -- position-2 acceptance
is already down to ~1%, so N=3 and N=4 spend real compute drafting tokens that are
almost never accepted, and the FP8 verifier's lower per-cycle base cost makes that
wasted overhead a larger proportional tax than it is for the slower BF16 verifier.
This differs from the reference run, which found N=1 optimal for the combined
config -- we did not assume that value transferred to our own draft head, and
sweeping confirmed it doesn't: N=1 underperforms our chosen N=2 by 47.5%.

**Note on the scoring rubric:** this result (`1468.70 tok/s`) does not clear the
`>1750 tok/s` threshold for the combined configuration, despite N=2 being the
genuine, measured optimum for this draft head and this hardware. See the final cell
for next steps on this gap.


## Central Question: Which Should Be Done First?

**Answer: Quantize the verifier first, then train the EAGLE-3 draft head on hidden
states generated by the quantized model -- but our reasoning for this differs from
the standard theoretical argument, because we tested that argument directly rather
than accepting it on paper.**

**Reasoning:**

The standard argument (train-on-what-you-serve) says: the EAGLE-3 draft head learns
`P(h_{t+1} | h_t)` for a specific verifier. If you train on BF16 hidden states and
then quantize the verifier to FP8, the draft head faces a train/serve distribution
shift and should lose acceptance rate. Training on FP8-generated hidden states
eliminates that shift by construction and should therefore give the highest
acceptance rate.

**We tested this directly instead of assuming it.** We trained a second EAGLE-3
draft head from scratch on hidden states generated by our *quantized* FP8 verifier
(same 3,000 samples, same layers `[2, 18, 33, 36]`, same hyperparameters -- only the
source verifier's precision differed), then benchmarked it against the FP8 verifier
at the same draft depth (N=2) as our original BF16-trained head.

**Result:** the FP8-trained draft head measured **1462.95 tok/s / 19.77% acceptance
rate** -- marginally *worse* than our original BF16-trained head's **1468.70 tok/s /
20.17%**. This is the opposite direction the theory predicts, though the gap
(-0.40 percentage points) is small enough to sit near our measurement noise floor.
It is reinforced by a second, independent piece of evidence: the two draft heads'
validation-loss curves matched within ~0.01 at every one of the 5 training epochs
(11.984 vs 11.994 final) -- they learned almost identically regardless of which
verifier generated their training data.

**Why the theory's predicted effect didn't show up:** we separately measured that
FP8 barely shifts the verifier's output distribution in the first place (a
direct BF16-vs-FP8 logit comparison on 3 test prompts found small, inconsistent-
direction differences in probability/entropy, and identical top-1 tokens in all 3
cases). If the input the draft head learns from barely changes between BF16 and
FP8, there is no large distribution-shift cost available for the "correct" training
order to avoid.

**So why still recommend quantizing first?** A different, independent argument that
*did* hold up under direct measurement: **speed**. Generating hidden states from the
FP8 verifier took 5 minutes 51 seconds for 3,000 samples, dramatically faster than
the original BF16 run. This reduces the cost of the offline pipeline's single most
time-consuming step, with no downside -- quantizing first is never worse for
quality (the acceptance-rate difference we measured is within noise either way),
and it is measurably faster to execute.

**Evidence summary from our own benchmark results:**
- Baseline: `838.74` tok/s
- Speculative decoding (BF16, N=2): `1069.82` tok/s (`+27.5%`)
- FP8 quantization: `1110.97` tok/s (`+32.5%`)
- FP8 + speculative decoding (N=2): `1468.70` tok/s (`+75.1%` vs baseline)
- The combined gain is *super-multiplicative*: naive multiplication of the two
  individual gains predicts `1417.05` tok/s; we measured `1468.70`, a further
  `+3.6%` on top of that. FP8 and EAGLE-3 both act on the same bottleneck (the
  verifier's forward pass) rather than independent parts of the decode cycle --
  confirmed by checking that the EAGLE-3 checkpoint is still `torch.bfloat16`
  (only the verifier was quantized) -- so shrinking that shared cost makes spec
  decoding's amortization more efficient, and the two techniques reinforce each
  other rather than simply stacking.

**Recommended pipeline:** BF16 model -> FP8 quantization -> hidden-state
generation (fast, confirmed) -> EAGLE-3 training -> serving. Order matters here
for a *speed* reason we measured directly, not the *acceptance-rate* reason the
theory predicted and our own controlled experiment did not confirm for this model.

**Known gap, being addressed next:** our absolute throughput on all three
scored configurations falls short of this assignment's scoring thresholds
(`>1250` / `>1550` / `>1750` tok/s), even though our baseline closely matches the
reference run's and our relative speedups are strong. This does not change any of
the reasoning above, but it is an open item we are investigating as a direct
follow-up to this submission.
